# Kaggle runner — E1 separated feature selection + classification (precomputed 5×5)

Input phải là version mới của `experiment-data-perm`, có `repeated_splits.csv` trong từng trong ba cohort. CSV chỉ định 25 outer train/test split (5 repeat × 5 fold). Feature selection chỉ thấy train; phân loại tune inner-CV trong train và chấm outer-test đúng một lần.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

INPUT_DATASET = Path('/kaggle/input/experiment-data-perm')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
COHORTS = ['gse68465_stage_early_late', 'tcga_luad_stage_early_late', 'tcga_lusc_stage_early_late']
K_GRID = [10, 50, 100, 200, 500]
RUN_FULL_CLASSIFIERS = True  # False = chỉ tạo panel; chạy phase B sau
RUN_WRAPPERS = False         # GA/BPSO là job tách riêng, bật ở lần chạy riêng


In [ ]:
# Gate 1: xác nhận input có đủ data và 25 split cố định/cohort trước khi chạy.
assert INPUT_DATASET.is_dir(), f'Không thấy {INPUT_DATASET}'
for cohort in COHORTS:
    root = INPUT_DATASET / cohort
    for filename in ('expression.h5', 'metadata.csv', 'repeated_splits.csv'):
        assert (root / filename).is_file(), f'Thiếu {cohort}/{filename}'
    split = pd.read_csv(root / 'repeated_splits.csv')
    assert set(split.role) == {'train', 'test'}
    assert split[['repeat', 'fold']].drop_duplicates().shape[0] == 25
    assert (split[split.role == 'test'].groupby(['repeat', 'sample_id']).size() == 1).all()
    assert not (split[split.role == 'train'].merge(split[split.role == 'test'], on=['repeat', 'fold', 'patient_id'])).shape[0]
print('INPUT_AND_SPLITS_OK')


In [ ]:
# Gate 2: clone đúng commit đã push, cài dependencies, rồi liên kết input read-only (không copy H5).
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
link = WORK_ROOT / 'experiment-data-perm'
if not link.exists(): link.symlink_to(INPUT_DATASET, target_is_directory=True)
assert link.resolve() == INPUT_DATASET.resolve()
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
print('COMMIT', commit_hash)


In [ ]:
# Phase A — filter + embedded. Không có classifier ở cell này.
subprocess.run([sys.executable, 'scripts/run_e1_feature_selection.py', '--datasets', *COHORTS, '--k', *map(str, K_GRID)], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Phase B — đọc fixed panels của Phase A; inner-CV chỉ dùng outer train.
if RUN_FULL_CLASSIFIERS:
    subprocess.run([sys.executable, 'scripts/run_e1_classification.py', '--datasets', *COHORTS], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Job RIÊNG cho GA/BPSO, sau đó Phase B chỉ dùng các panel wrapper đã lưu.
if RUN_WRAPPERS:
    subprocess.run([sys.executable, 'scripts/run_e1_wrappers_all_cohorts.py'], cwd=REPO_DIR, env=os.environ.copy(), check=True)
    subprocess.run([sys.executable, 'scripts/run_e1_classification.py', '--datasets', *COHORTS, '--methods', 'ga', 'bpso', '--selection-root', str(WORK_ROOT / 'results/survey_v2/E1_wrapper_feature_selection'), '--out', 'E1_wrapper_classification'], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Gate 3: chỉ save/download output nếu provenance đầy đủ.
result_root = WORK_ROOT / 'results/survey_v2'
configs = list(result_root.glob('E1_*/*/config.json'))
assert configs, 'Không tìm thấy config output'
for path in configs:
    cfg = json.loads(path.read_text(encoding='utf-8'))
    assert cfg.get('traceable') is True and cfg.get('commit_hash') == commit_hash, path
print('E1_OK', result_root)
